In [1]:
import sys
print(sys.executable)

/Users/joe/coding/CSE188/flight-telemetry-NLP/venv/bin/python


In [16]:
## Creating Telemetry Data
import json
import pandas as pd
from pathlib import Path

Path("../data").mkdir(exist_ok=True)

examples = [
    {
        "id": "ex01",
        "difficulty": "easy",
        "scenario": "United departure climbing east-southeast out of SFO",
        "raw_input": {
            "icao24": "a8c1f2",
            "startTime": 1713300000,
            "endTime": 1713300360,
            "callsign": "UAL245 ",
            "path": [
                [1713300000, 37.6188, -122.3754, 250.0, 102.0, False],
                [1713300120, 37.6205, -122.3000, 850.0, 105.0, False],
                [1713300240, 37.6220, -122.2200, 1550.0, 108.0, False],
                [1713300360, 37.6240, -122.1300, 2450.0, 112.0, False]
            ],
            "derived_metadata": {
                "alert": "climb",
                "notes": "steady eastbound departure from SFO"
            }
        },
        "gold_summary": "UAL245 is climbing steadily after departure from SFO while tracking generally east-southeast.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude increases steadily",
            "heading remains generally east-southeast",
            "trajectory is consistent with departure",
            "no ground contact"
        ]
    },
    {
        "id": "ex02",
        "difficulty": "easy",
        "scenario": "American arrival descending south toward SFO",
        "raw_input": {
            "icao24": "ab12cd",
            "startTime": 1713301000,
            "endTime": 1713301360,
            "callsign": "AAL187 ",
            "path": [
                [1713301000, 37.7800, -122.3600, 3100.0, 185.0, False],
                [1713301120, 37.7300, -122.3550, 2400.0, 184.0, False],
                [1713301240, 37.6800, -122.3520, 1500.0, 183.0, False],
                [1713301360, 37.6320, -122.3500, 700.0, 182.0, False]
            ],
            "derived_metadata": {
                "alert": "rapid_descent",
                "notes": "southbound arrival consistent with SFO approach flow"
            }
        },
        "gold_summary": "AAL187 is descending rapidly on a southbound approach toward SFO.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude decreases rapidly",
            "heading remains nearly south",
            "trajectory is consistent with arrival",
            "rapid descent alert is present"
        ]
    },
    {
        "id": "ex03",
        "difficulty": "easy",
        "scenario": "Delta cruise segment eastbound over central California",
        "raw_input": {
            "icao24": "c3d4e5",
            "startTime": 1713302000,
            "endTime": 1713302360,
            "callsign": "DAL412 ",
            "path": [
                [1713302000, 36.8000, -121.9000, 10850.0, 91.0, False],
                [1713302120, 36.8010, -121.7900, 10860.0, 92.0, False],
                [1713302240, 36.8015, -121.6800, 10840.0, 90.0, False],
                [1713302360, 36.8020, -121.5700, 10855.0, 93.0, False]
            ],
            "derived_metadata": {
                "alert": "level_flight",
                "notes": "stable eastbound cruise"
            }
        },
        "gold_summary": "DAL412 remains in near-level cruise while continuing eastbound with minimal heading change.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude stays nearly constant",
            "heading remains close to east",
            "trajectory is stable",
            "level flight alert is present"
        ]
    },
    {
        "id": "ex04",
        "difficulty": "medium",
        "scenario": "Southwest aircraft descending while turning right toward OAK approach corridor",
        "raw_input": {
            "icao24": "d6e7f8",
            "startTime": 1713303000,
            "endTime": 1713303360,
            "callsign": "SWA882 ",
            "path": [
                [1713303000, 37.9100, -122.5800, 3900.0, 105.0, False],
                [1713303120, 37.8850, -122.5100, 3200.0, 128.0, False],
                [1713303240, 37.8450, -122.4550, 2400.0, 152.0, False],
                [1713303360, 37.8000, -122.4200, 1650.0, 172.0, False]
            ],
            "derived_metadata": {
                "alert": "descending_turn",
                "notes": "descent with pronounced right turn toward East Bay"
            }
        },
        "gold_summary": "SWA882 is descending while making a strong right turn, shifting from an easterly heading toward nearly southbound flight.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude decreases steadily",
            "heading turns right significantly",
            "final heading is near south",
            "descent and turn occur together"
        ]
    },
    {
        "id": "ex05",
        "difficulty": "medium",
        "scenario": "Alaska departure climbing and turning left out of SJC",
        "raw_input": {
            "icao24": "e9f0a1",
            "startTime": 1713304000,
            "endTime": 1713304360,
            "callsign": "ASA671 ",
            "path": [
                [1713304000, 37.3639, -121.9289, 280.0, 330.0, False],
                [1713304120, 37.4200, -121.9650, 900.0, 312.0, False],
                [1713304240, 37.4900, -122.0100, 1650.0, 292.0, False],
                [1713304360, 37.5650, -122.0700, 2550.0, 275.0, False]
            ],
            "derived_metadata": {
                "alert": "climbing_turn",
                "notes": "departure climb with left turn from SJC toward the peninsula"
            }
        },
        "gold_summary": "ASA671 is climbing after departure from SJC while turning left from a northwest heading toward a more westerly track.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude increases steadily",
            "heading turns left",
            "climb and turn occur simultaneously",
            "trajectory is consistent with departure"
        ]
    },
    {
        "id": "ex06",
        "difficulty": "hard",
        "scenario": "JetBlue aircraft in holding-pattern-like maneuver west of SFO",
        "raw_input": {
            "icao24": "f2a3b4",
            "startTime": 1713305000,
            "endTime": 1713305600,
            "callsign": "JBU516 ",
            "path": [
                [1713305000, 37.6900, -122.6200, 3200.0, 45.0, False],
                [1713305150, 37.7350, -122.5750, 3190.0, 120.0, False],
                [1713305300, 37.7050, -122.5200, 3185.0, 205.0, False],
                [1713305450, 37.6550, -122.5550, 3175.0, 292.0, False],
                [1713305600, 37.6850, -122.6200, 3180.0, 18.0, False]
            ],
            "derived_metadata": {
                "alert": "holding_pattern",
                "notes": "looping track with nearly constant altitude"
            }
        },
        "gold_summary": "JBU516 appears to be in a holding pattern, circling west of SFO at nearly constant altitude.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude remains nearly constant",
            "heading changes dramatically over the segment",
            "trajectory loops back toward its starting area",
            "holding pattern alert is present"
        ]
    },
    {
        "id": "ex07",
        "difficulty": "medium",
        "scenario": "Frontier final approach aligned toward SFO",
        "raw_input": {
            "icao24": "0a1b2c",
            "startTime": 1713306000,
            "endTime": 1713306360,
            "callsign": "FFT302 ",
            "path": [
                [1713306000, 37.7350, -122.3550, 1900.0, 182.0, False],
                [1713306120, 37.6900, -122.3535, 1350.0, 181.0, False],
                [1713306240, 37.6500, -122.3525, 800.0, 181.0, False],
                [1713306360, 37.6150, -122.3515, 280.0, 180.0, False]
            ],
            "derived_metadata": {
                "alert": "final_approach",
                "notes": "stable approach with constant southerly alignment"
            }
        },
        "gold_summary": "FFT302 is on a stable final approach, descending steadily while maintaining a nearly due-south alignment.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude decreases steadily",
            "heading remains stable",
            "trajectory suggests final approach alignment",
            "final approach alert is present"
        ]
    },
    {
        "id": "ex08",
        "difficulty": "medium",
        "scenario": "United taxi or rollout after landing at SFO",
        "raw_input": {
            "icao24": "1d2e3f",
            "startTime": 1713307000,
            "endTime": 1713307360,
            "callsign": "UAL903 ",
            "path": [
                [1713307000, 37.6150, -122.3900, 6.0, 110.0, True],
                [1713307120, 37.6140, -122.3865, 6.0, 95.0, True],
                [1713307240, 37.6130, -122.3830, 6.0, 80.0, True],
                [1713307360, 37.6122, -122.3795, 6.0, 65.0, True]
            ],
            "derived_metadata": {
                "alert": "on_ground",
                "notes": "surface movement after landing"
            }
        },
        "gold_summary": "UAL903 is on the ground and appears to be taxiing after landing, with negligible altitude change and gradual heading adjustments.",
        "gold_facts": [
            "aircraft is on the ground",
            "altitude stays near zero",
            "movement continues on the surface",
            "heading changes while taxiing",
            "on-ground alert is present"
        ]
    },
    {
        "id": "ex09",
        "difficulty": "hard",
        "scenario": "American aircraft with unstable altitude profile over the East Bay",
        "raw_input": {
            "icao24": "2f3a4b",
            "startTime": 1713308000,
            "endTime": 1713308480,
            "callsign": "AAL560 ",
            "path": [
                [1713308000, 37.9500, -122.2500, 2550.0, 118.0, False],
                [1713308120, 37.9200, -122.1800, 2875.0, 120.0, False],
                [1713308240, 37.8900, -122.1050, 2420.0, 124.0, False],
                [1713308360, 37.8550, -122.0250, 2760.0, 128.0, False],
                [1713308480, 37.8200, -121.9450, 2230.0, 132.0, False]
            ],
            "derived_metadata": {
                "alert": "unstable_profile",
                "notes": "alternating climbs and descents on a generally east-southeast path"
            }
        },
        "gold_summary": "AAL560 shows an unstable altitude profile, alternating between climbs and descents while continuing generally east-southeast.",
        "gold_facts": [
            "aircraft is airborne",
            "altitude does not follow a single monotonic trend",
            "there are alternating climbs and descents",
            "heading changes are modest",
            "unstable profile alert is present"
        ]
    },
    {
        "id": "ex10",
        "difficulty": "hard",
        "scenario": "Delta aircraft near level at first then entering abrupt descent south of Monterey Bay",
        "raw_input": {
            "icao24": "3c4d5e",
            "startTime": 1713309000,
            "endTime": 1713309480,
            "callsign": "DAL155 ",
            "path": [
                [1713309000, 36.5500, -121.9000, 5400.0, 92.0, False],
                [1713309120, 36.5520, -121.8100, 5390.0, 93.0, False],
                [1713309240, 36.5530, -121.7200, 5385.0, 95.0, False],
                [1713309360, 36.5250, -121.6400, 4450.0, 98.0, False],
                [1713309480, 36.4900, -121.5600, 3180.0, 102.0, False]
            ],
            "derived_metadata": {
                "alert": "abrupt_descent",
                "notes": "stable segment followed by sharp drop in altitude"
            }
        },
        "gold_summary": "DAL155 remains nearly level at first and then enters an abrupt descent while continuing generally eastward.",
        "gold_facts": [
            "aircraft is airborne",
            "initial segment is near level",
            "later segment shows a sharp descent",
            "heading remains generally east",
            "abrupt descent alert is present"
        ]
    }
]

# Save master JSON
with open("../data/telemetry_examples.json", "w") as f:
    json.dump(examples, f, indent=2)

# Save convenience CSV
rows = []
for ex in examples:
    rows.append({
        "id": ex["id"],
        "difficulty": ex["difficulty"],
        "scenario": ex["scenario"],
        "icao24": ex["raw_input"]["icao24"],
        "callsign": ex["raw_input"]["callsign"].strip(),
        "startTime": ex["raw_input"]["startTime"],
        "endTime": ex["raw_input"]["endTime"],
        "alert": ex["raw_input"]["derived_metadata"]["alert"],
        "notes": ex["raw_input"]["derived_metadata"]["notes"],
        "gold_summary": ex["gold_summary"],
        "gold_facts": " | ".join(ex["gold_facts"]),
        "raw_input_json": json.dumps(ex["raw_input"])
    })

df = pd.DataFrame(rows)
df.to_csv("../data/telemetry_examples.csv", index=False)

print("Saved:")
print("- ../data/telemetry_examples.json")
print("- ../data/telemetry_examples.csv")

df.head()

Saved:
- ../data/telemetry_examples.json
- ../data/telemetry_examples.csv


,id,difficulty,scenario,icao24,callsign,startTime,endTime,alert,notes,gold_summary,gold_facts,raw_input_json
0,ex01,easy,United departure climbing east-southeast out o...,a8c1f2,UAL245,1713300000,1713300360,climb,steady eastbound departure from SFO,UAL245 is climbing steadily after departure fr...,aircraft is airborne | altitude increases stea...,"{""icao24"": ""a8c1f2"", ""startTime"": 1713300000, ..."
1,ex02,easy,American arrival descending south toward SFO,ab12cd,AAL187,1713301000,1713301360,rapid_descent,southbound arrival consistent with SFO approac...,AAL187 is descending rapidly on a southbound a...,aircraft is airborne | altitude decreases rapi...,"{""icao24"": ""ab12cd"", ""startTime"": 1713301000, ..."
2,ex03,easy,Delta cruise segment eastbound over central Ca...,c3d4e5,DAL412,1713302000,1713302360,level_flight,stable eastbound cruise,DAL412 remains in near-level cruise while cont...,aircraft is airborne | altitude stays nearly c...,"{""icao24"": ""c3d4e5"", ""startTime"": 1713302000, ..."
3,ex04,medium,Southwest aircraft descending while turning ri...,d6e7f8,SWA882,1713303000,1713303360,descending_turn,descent with pronounced right turn toward East...,SWA882 is descending while making a strong rig...,aircraft is airborne | altitude decreases stea...,"{""icao24"": ""d6e7f8"", ""startTime"": 1713303000, ..."
4,ex05,medium,Alaska departure climbing and turning left out...,e9f0a1,ASA671,1713304000,1713304360,climbing_turn,departure climb with left turn from SJC toward...,ASA671 is climbing after departure from SJC wh...,aircraft is airborne | altitude increases stea...,"{""icao24"": ""e9f0a1"", ""startTime"": 1713304000, ..."


In [17]:
from pathlib import Path
import json

data_path = Path("../data/telemetry_examples.json")
print("Exists:", data_path.exists())
print("Size:", data_path.stat().st_size)

with open(data_path, "r") as f:
    examples = json.load(f)

print("Loaded examples:", len(examples))
print("First example ID:", examples[0]["id"])
print("First example callsign:", examples[0]["raw_input"]["callsign"])

Exists: True
Size: 13423
Loaded examples: 10
First example ID: ex01
First example callsign: UAL245 


In [10]:
## Connection test

from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)

response = client.chat.completions.create(
    model="local-model",
    messages=[
        {
            "role": "user",
            "content": """You are a helpful assistant.
Return only JSON with keys altitude_trend and heading_trend for this:
altitude rises from 300 to 2000, heading shifts east."""
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

 {
  "altitude_trend": "increase",
  "heading_trend": "east"
}


In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)

def call_model(prompt_text, instruction_prefix=None, temperature=0):
    if instruction_prefix:
        full_prompt = instruction_prefix.strip() + "\n\n" + prompt_text.strip()
    else:
        full_prompt = prompt_text.strip()

    response = client.chat.completions.create(
        model="local-model",
        messages=[
            {"role": "user", "content": full_prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

In [18]:
import json
from pathlib import Path

print("Current working directory:", Path.cwd())

with open("../data/telemetry_examples.json", "r") as f:
    examples = json.load(f)

def load_prompt(path):
    with open(path, "r") as f:
        return f.read()

baseline_direct_prompt = load_prompt("../prompts/baseline_direct.txt")
baseline_structured_prompt = load_prompt("../prompts/baseline_structured.txt")
extractor_prompt = load_prompt("../prompts/agent_extractor.txt")
verifier_prompt = load_prompt("../prompts/agent_verifier.txt")
writer_prompt = load_prompt("../prompts/agent_writer.txt")
judge_prompt = load_prompt("../prompts/agent_judge.txt")

print(f"Loaded {len(examples)} examples.")
print("Prompt files loaded successfully.")

Current working directory: /Users/joe/coding/CSE188/flight-telemetry-NLP/notebooks
Loaded 10 examples.
Prompt files loaded successfully.


In [19]:
example = examples[0]

raw_input_str = json.dumps(example["raw_input"], indent=2)

filled_prompt = baseline_direct_prompt.format(raw_input=raw_input_str)

print(filled_prompt[:1200])

You are an aviation telemetry summarization assistant.

Your task is to read a structured flight telemetry snippet and write a concise 1-2 sentence summary of what is happening.

Requirements:
- Use only information supported by the telemetry.
- Do not invent events, causes, or destinations.
- Focus on altitude trend, heading trend, ground/air status, and any alert metadata if present.
- Be concise and factual.

Telemetry input:
{
  "icao24": "a8c1f2",
  "startTime": 1713300000,
  "endTime": 1713300360,
  "callsign": "UAL245 ",
  "path": [
    [
      1713300000,
      37.6188,
      -122.3754,
      250.0,
      102.0,
      false
    ],
    [
      1713300120,
      37.6205,
      -122.3,
      850.0,
      105.0,
      false
    ],
    [
      1713300240,
      37.622,
      -122.22,
      1550.0,
      108.0,
      false
    ],
    [
      1713300360,
      37.624,
      -122.13,
      2450.0,
      112.0,
      false
    ]
  ],
  "derived_metadata": {
    "alert": "climb",
    "no

In [20]:
baseline_output = call_model(
    filled_prompt,
    instruction_prefix="You are a precise aviation telemetry summarization assistant.",
    temperature=0
)

print(baseline_output)

 Aircraft UAL245 is steadily climbing and moving eastward, departing from San Francisco International Airport (SFO). An alert for a climb has been issued.


In [21]:
extractor_filled = extractor_prompt.format(
    raw_input=raw_input_str
)

print("EXTRACTOR PROMPT PREVIEW:\n")
print(extractor_filled[:1500])

EXTRACTOR PROMPT PREVIEW:

You are the Extractor Agent in a multi-agent aviation telemetry system.

Your job is to extract atomic facts from the telemetry input.
Do not write a narrative summary.
Do not speculate.
Do not infer destinations or intentions unless directly supported by the telemetry.

Return your output as a JSON object with these keys:
- callsign
- airborne_status
- altitude_trend
- heading_trend
- movement_pattern
- alert
- atomic_facts

Guidelines:
- "airborne_status" should be either "airborne" or "on_ground".
- "altitude_trend" should be one of: increasing, decreasing, stable, mixed.
- "heading_trend" should describe the overall pattern, such as stable eastbound, turning right, turning left, highly variable.
- "movement_pattern" should describe the best-supported motion pattern, such as climb, descent, final approach, holding pattern, taxi, level flight, or mixed.
- "alert" should copy the alert metadata if present.
- "atomic_facts" should be a list of short factual s

In [22]:
extractor_output = call_model(
    extractor_filled,
    instruction_prefix="You are a precise extraction agent that returns structured JSON only.",
    temperature=0
)

print("EXTRACTOR OUTPUT:\n")
print(extractor_output)

KeyboardInterrupt: 